# 02 · Silver — Cleaning, Conforming & Daily Derivation

Turn messy bronze into **clean, typed, deduplicated** tables, and **derive the daily
consumption table from the half-hourly readings** (so we don't need the pre-built daily
dataset). (Brief **§3 Pre-processing**.)

Problems fixed here:
* `energy(kWh/hh)` has leading spaces and literal `"Null"` text → trim + coerce to null.
* timestamps `2012-10-12 00:30:00.0000000` have 7 fractional digits Spark won't parse →
  keep the first 19 chars.
* derive a `half_hour` index 0–47 (00:00 … 23:30) for load-profile & clustering.
* ACORN blanks → `"Unclassified"`; duplicates dropped.

In [ ]:
%run ./00_config_and_setup

In [ ]:
from pyspark.sql import functions as F

def save_silver(df, name):
    tgt = table("silver", name)
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true").saveAsTable(tgt)
    print(f"silver {name:<16} rows={spark.table(tgt).count():>12,}  cols={len(df.columns)}")
    return tgt

## 1. Half-hourly readings — clean + derive time features
This is the foundation table; `silver_daily` and the load profile are built from it.

In [ ]:
hh = (spark.table(table("bronze", "halfhourly"))
    .withColumnRenamed("energy_kWh_hh", "energy_kwh")   # bronze sanitised energy(kWh/hh)
    .withColumn("energy_kwh", F.expr("try_cast(trim(energy_kwh) AS double)"))  # "Null"/spaces -> null (ANSI-safe)
    .withColumn("tstp", F.expr("try_to_timestamp(substring(tstp, 1, 19))"))  # ANSI-safe; drop 7-digit fraction
    .filter(F.col("tstp").isNotNull() & F.col("energy_kwh").isNotNull())
    .withColumn("date", F.to_date("tstp"))
    .withColumn("hour", F.hour("tstp"))
    # half_hour index: 0..47  (e.g. 00:00->0, 00:30->1, ... 23:30->47)
    .withColumn("half_hour", F.col("hour") * 2 + (F.minute("tstp") >= 30).cast("int"))
    .dropDuplicates(["LCLid", "tstp"])
    .select("LCLid", "tstp", "date", "hour", "half_hour", "energy_kwh"))

save_silver(hh, "halfhourly")

## 2. Daily consumption — DERIVED from half-hourly
`groupBy(household, day)` reproduces the standard daily statistics (sum/mean/median/…),
so the rest of the pipeline is identical to using a pre-aggregated daily dataset.

In [ ]:
daily = (spark.table(table("silver", "halfhourly"))
    .groupBy("LCLid", F.col("date").alias("day"))
    .agg(F.round(F.sum("energy_kwh"), 4).alias("energy_sum"),
         F.round(F.avg("energy_kwh"), 5).alias("energy_mean"),
         F.expr("percentile_approx(energy_kwh, 0.5)").alias("energy_median"),
         F.max("energy_kwh").alias("energy_max"),
         F.min("energy_kwh").alias("energy_min"),
         F.round(F.stddev("energy_kwh"), 5).alias("energy_std"),
         F.count("energy_kwh").alias("energy_count")))

save_silver(daily, "daily")

## 3. Households dimension (tariff + ACORN affluence group)

In [ ]:
households = (spark.table(table("bronze", "households"))
    .select("LCLid", "stdorToU", "Acorn", "Acorn_grouped")
    .withColumn("tariff", F.when(F.col("stdorToU") == "ToU", F.lit("Time-of-Use"))
                           .otherwise(F.lit("Standard")))
    .withColumn("acorn_group",
                F.when(F.col("Acorn_grouped").isin("ACORN-", "ACORN-U", ""), F.lit("Unclassified"))
                 .otherwise(F.col("Acorn_grouped")))
    .withColumnRenamed("Acorn", "acorn_code")
    .dropDuplicates(["LCLid"]))

save_silver(households, "households")

## 4. Weather (daily)

In [ ]:
w_num = ["temperatureMax", "temperatureMin", "humidity", "windSpeed",
         "cloudCover", "pressure", "uvIndex"]
weather = spark.table(table("bronze", "weather_daily")).withColumn("date", F.to_date("time"))
for c in w_num:
    if c in weather.columns:
        weather = weather.withColumn(c, F.expr(f"try_cast(`{c}` AS double)"))  # ANSI-safe
weather = (weather
    .withColumn("temp_avg", (F.col("temperatureMax") + F.col("temperatureMin")) / 2.0)
    .select("date", "temp_avg", "temperatureMax", "temperatureMin",
            "humidity", "windSpeed", "cloudCover", "pressure", "uvIndex",
            "precipType", "summary")
    .dropDuplicates(["date"]))

save_silver(weather, "weather_daily")

## 5. Bank holidays (calendar dimension)

In [ ]:
holidays = (spark.table(table("bronze", "bank_holidays"))
    .withColumnRenamed("Bank_holidays", "date")   # bronze sanitised "Bank holidays"
    .withColumn("date", F.to_date("date"))
    .withColumnRenamed("Type", "holiday_name")
    .filter(F.col("date").isNotNull())
    .dropDuplicates(["date"]))

save_silver(holidays, "bank_holidays")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SILVER}"))